# BACKTESTING_272 — Stratégie Momentum Dual Signal Weight
 
**Univers** : STOXX Europe 600  
**Période** : 2016-12-30 → 2026-02-20  
**Benchmark** : ESTRON Index


## 1. Importation des Libraries <a id='1'></a>

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

import pandas as pd
import numpy as np
from IPython.display import display


from backtesting import DataLoader, BacktestEngine, ReportingEngine


## 2. Paramètres du portefeuille <a id='1'></a>

In [ ]:
INITIAL_CAPITAL = 1_000_000

TRANSACTION_COST = 0.0005

SELECTION_QUANTILE = 0.40

METHODS = ["signal_weight"]

"""
- SECTOR_MIN : nombre minimum de poids par secteur
- SECTOR_MAX : nombre maximum de poids par secteur
"""
SECTOR_MIN = None
SECTOR_MAX = None

print(f"Capital initial   : {INITIAL_CAPITAL:,.0f} €")
print(f"Frais transaction : {TRANSACTION_COST * 10_000:.1f} bps")
print(f"Sélection         : top/bottom {SELECTION_QUANTILE*100:.0f}%")
print(f"Méthodes          : {METHODS}")
print(f"Contraintes       : SECTOR_MIN={SECTOR_MIN}, SECTOR_MAX={SECTOR_MAX}")

Capital initial   : 1,000,000 €
Frais transaction : 5.0 bps
Sélection         : top/bottom 40%
Méthodes          : ['signal_weight']
Contraintes       : SECTOR_MIN=None, SECTOR_MAX=None



## 3. Chargement des données <a id='1'></a>

In [ ]:
"""
- start_date : date de début du backtest (format "YYYY-MM-DD")
- end_date : date de fin du backtest (format "YYYY-MM-DD")
"""

loader = DataLoader(start_date="2016-12-30", end_date="2026-02-20")

"""
- initial_capital    : implémente le capital initinial du portefeuille
- transaction_cost   : implémente le coût de transaction par trade 
- selection_quantile : implémente le quantile utilisé pour sélectionner les positions long/short au travers des signaux
- sector_min         : implémente le poids minimum autorisé par secteur (contrainte de diversification)
- sector_max         : implémente le poids maximum autorisé par secteur (contrainte de diversification)
"""

engine = BacktestEngine(
    data_loader=loader,
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=TRANSACTION_COST,
    selection_quantile=SELECTION_QUANTILE,
    sector_min=SECTOR_MIN,
    sector_max=SECTOR_MAX,
)

results = {}

print(f"\nDonnées chargées : {len(loader.rebalancing_dates)} dates de rebalancement")
print(f"Tickers disponibles : {loader.prices.shape[1]}")
print(f"Période des prix : {loader.prices.index[0].date()} → {loader.prices.index[-1].date()}")

[DataLoader] Chargement des données en cours...
[DataLoader] Prêt : 111 dates de rebalancement (2016-12-30 → 2026-02-20).

Données chargées : 111 dates de rebalancement
Tickers disponibles : 980
Période des prix : 2015-09-01 → 2026-02-20



## 4. Backtesting <a id='1'></a>

In [ ]:
"""
- Choix : signal_weight : implémente une allocation basée sur des poids proportionnels à la force du signal de momentum
"""
METHOD = "signal_weight"

if METHOD in METHODS:
    print(f"{'='*60}")
    print(f"  Lancement : {METHOD}")
    print(f"{'='*60}")
    results[METHOD] = engine.run(METHOD)
    nav_final = results[METHOD].nav.iloc[-1]
    total_ret = nav_final / INITIAL_CAPITAL - 1
    print(f"\n  NAV finale : {nav_final:,.0f} €  |  Rendement : {total_ret:.2%}")
else:
    print(f"'{METHOD}' non sélectionné dans METHODS — cellule ignorée.")

  Lancement : signal_weight

[BacktestEngine] Démarrage — méthode : signal_weight
  [  1/111] 2016-12-30 NAV=   1,021,856€  Long=232  Short=232
  [  2/111] 2017-01-31 NAV=   1,012,982€  Long=230  Short=230
  [  3/111] 2017-02-28 NAV=   1,013,346€  Long=231  Short=231
  [  4/111] 2017-03-31 NAV=   1,022,235€  Long=230  Short=230
  [  5/111] 2017-04-28 NAV=   1,007,083€  Long=230  Short=230
  [  6/111] 2017-05-31 NAV=   1,027,083€  Long=229  Short=229
  [  7/111] 2017-06-30 NAV=   1,048,749€  Long=231  Short=231
  [  8/111] 2017-07-31 NAV=   1,083,639€  Long=231  Short=231
  [  9/111] 2017-08-31 NAV=   1,081,710€  Long=231  Short=231
  [ 10/111] 2017-09-29 NAV=   1,105,529€  Long=230  Short=230
  [ 11/111] 2017-10-31 NAV=   1,116,142€  Long=203  Short=203
  [ 12/111] 2017-11-30 NAV=   1,102,645€  Long=232  Short=232
  [ 13/111] 2017-12-29 NAV=   1,132,476€  Long=231  Short=231
  [ 14/111] 2018-01-31 NAV=   1,113,214€  Long=232  Short=232
  [ 15/111] 2018-02-28 NAV=   1,140,574€  Long=167


## 5. Reporting <a id='1'></a>

In [ ]:
METHOD_TO_REPORT = METHODS.pop(0) 

"""
- AS_OF_DATE       : date d’arrêté des métriques de performance (format "YYYY-MM-DD")
"""
AS_OF_DATE = "2026-01-30"

reporter = ReportingEngine(results[METHOD_TO_REPORT], loader)
as_of = pd.Timestamp(AS_OF_DATE)

In [ ]:

metrics = reporter.compute_metrics(as_of)


def format_metric(v, key=None):
    """
    - si la valeur est "--", elle est retournée telle qu'une valeur manquante,
    - si la valeur est un float :
         si la métrique est un ratio (Sharpe, IR, Corrélation, etc.) -> format décimal
         sinon (rendements, vol, drawdown, TC%) -> format en %
    - sinon, la valeur est convertie en chaîne de caractères.
    """
    if v == "--":
        return "--"

    if isinstance(v, float):

        if key is not None and ("(€)" in key or "€" in key):
            return f"{v:,.2f}"


        ratio_keys = (
            "Sharpe", "Sortino", "Information Ratio",
            "Calmar", "Corrélation", "Tracking Error"
        )
        if key is not None and any(rk in key for rk in ratio_keys):
            return f"{v:.4f}"


        return f"{v*100:.2f}%"

    return str(v)


"""
- Serie pandas indéxée par le nom des indicateurs et datée à la date de reporting
"""

metrics_series = pd.Series({
    k: format_metric(v, k) for k, v in metrics.items()
}, name=str(as_of.date()))
metrics_series.index.name = "Métrique"

categories = {
    "Performance": [
        "Rendement cumulé (période)", "Rendement cumulé 1 an",
        "Rendement cumulé 2 ans", "Rendement cumulé YTD", "Rendement cumulé MTD",
        "Rendement annualisé (période)",
        "CAGR (période)", "CAGR 1 an", "CAGR 2 ans", "CAGR YTD", "CAGR MTD",
    ]}

for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>12}")


──────────────────────────────────────────────────
  Performance
──────────────────────────────────────────────────
  Rendement cumulé (période)                          94.10%
  Rendement cumulé 1 an                               19.63%
  Rendement cumulé 2 ans                              34.73%
  Rendement cumulé YTD                                 2.12%
  Rendement cumulé MTD                                 2.12%
  Rendement annualisé (période)                        8.17%
  CAGR (période)                                       6.80%
  CAGR 1 an                                           19.79%
  CAGR 2 ans                                          17.11%
  CAGR YTD                                            29.03%
  CAGR MTD                                            29.03%


In [ ]:
categories = {
    "Risque & Ratios": [
        "Sharpe ratio (période)", "Sharpe ratio 1 an",
        "Sortino ratio (période)", "Sortino ratio 1 an",
        "Tracking Error (période)", "Tracking Error 1 an", "Tracking Error 3 ans",
        "Information Ratio (période)", "Information Ratio 1 an",
        "Information Ratio 2 ans",
        "Calmar ratio (période)",
        "VaR 95% (période)", "VaR 95% 1 an",
        "CVaR 95% (période)", "CVaR 95% 1 an",
        "Volatilité (période)", "Volatilité 1 an", "Volatilité 2 ans",
        "Max Drawdown (période)", "Max Drawdown 1 an",
        "Max Drawdown 2 ans", "Max Drawdown YTD", "Max Drawdown MTD",
    ]}


for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>20}")


──────────────────────────────────────────────────
  Risque & Ratios
──────────────────────────────────────────────────
  Sharpe ratio (période)                                      0.5328
  Sharpe ratio 1 an                                           1.3564
  Sortino ratio (période)                                     0.5866
  Sortino ratio 1 an                                          1.8509
  Tracking Error (période)                                    0.1400
  Tracking Error 1 an                                         0.1233
  Tracking Error 3 ans                                        0.1081
  Information Ratio (période)                                 0.5328
  Information Ratio 1 an                                      1.3564
  Information Ratio 2 ans                                     1.2420
  Calmar ratio (période)                                      0.3505
  VaR 95% (période)                                            1.28%
  VaR 95% 1 an                                     

In [ ]:

categories = {
       "Benchmark (ESTER)": [
        "Corrélation avec benchmark (période)",
        "Benchmark - Rendement cumulé (période)",
        "Benchmark - Rendement cumulé 1 an",
        "Benchmark - Rendement cumulé 2 ans",
        "Benchmark - Rendement cumulé YTD",
        "Benchmark - Rendement cumulé MTD",
        "Benchmark - Rendement annualisé (période)",
    ],
    "Coûts de transaction": ["TC total (€)", "TC total (%)"]}

for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>12}")


──────────────────────────────────────────────────
  Benchmark (ESTER)
──────────────────────────────────────────────────
  Corrélation avec benchmark (période)                0.0001
  Benchmark - Rendement cumulé (période)               8.38%
  Benchmark - Rendement cumulé 1 an                    2.15%
  Benchmark - Rendement cumulé 2 ans                   5.92%
  Benchmark - Rendement cumulé YTD                     0.16%
  Benchmark - Rendement cumulé MTD                     0.16%
  Benchmark - Rendement annualisé (période)            1.25%

──────────────────────────────────────────────────
  Coûts de transaction
──────────────────────────────────────────────────
  TC total (€)                                     73,687.37
  TC total (%)                                         7.37%


In [ ]:
all_metrics_df = reporter.compute_all_metrics()
display(all_metrics_df.tail(5))

,Rendement cumulé (période),Rendement cumulé 1 an,Rendement cumulé 2 ans,Rendement cumulé YTD,Rendement cumulé MTD,Rendement annualisé (période),CAGR (période),CAGR 1 an,CAGR 2 ans,CAGR YTD,...,Corrélation avec benchmark (période),Benchmark - Rendement cumulé (période),Benchmark - Rendement cumulé 1 an,Benchmark - Rendement cumulé 2 ans,Benchmark - Rendement cumulé YTD,Benchmark - Rendement cumulé MTD,Benchmark - Rendement annualisé (période),TC total (€),TC total (%),PnL (€)
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-31,0.809723,0.123784,0.2309,0.12099,0.007695,0.075893,0.061839,0.124356,0.112755,0.147085,...,-0.001788,0.078571,0.0244,0.064479,0.019022,0.00166,0.01222,70251.977846,0.070252,716455.810478
2025-11-28,0.83362,0.127731,0.256481,0.135384,0.01284,0.076768,0.062774,0.130488,0.13041,0.149914,...,-0.001488,0.080191,0.023416,0.062846,0.020552,0.001501,0.012304,70877.716518,0.070878,738494.833608
2025-12-31,0.89895,0.175163,0.377207,0.175163,0.035036,0.079951,0.066184,0.173406,0.177122,0.175293,...,-0.000455,0.082104,0.022359,0.061158,0.022359,0.001771,0.012421,71909.068205,0.071909,799404.426651
2026-01-30,0.941004,0.196272,0.347285,0.021158,0.021158,0.081659,0.067983,0.197863,0.171096,0.290342,...,0.000096,0.083847,0.021518,0.059186,0.001611,0.001611,0.012511,73687.374059,0.073687,837475.592603
2026-02-20,0.962451,0.196835,0.334433,0.032103,0.010719,0.082482,0.068758,0.194436,0.162898,0.253955,...,0.000355,0.085069,0.021045,0.057965,0.00274,0.001127,0.012569,74294.974297,0.074295,857171.179007



## 5. Graphiques <a id='1'></a>

### 5a. Rendements cumulés — Portefeuille vs Benchmark

In [ ]:
fig_cumul = reporter.plot_cumulative_returns()
fig_cumul.show()

### 5b. Drawdowns du portefeuille

In [ ]:
fig_dd = reporter.plot_drawdowns()
fig_dd.show()

### 5c. Volatilité glissante annualisée

In [ ]:
fig_vol = reporter.plot_historical_volatility()
fig_vol.show()

### 5d. Corrélation glissante avec le Benchmark

In [ ]:
fig_corr = reporter.plot_historical_correlation()
fig_corr.show()

### 5e. Composition du portefeuille

In [ ]:

comp_date = pd.Timestamp(AS_OF_DATE)


comp_df = reporter.get_portfolio_composition(comp_date)
print(f"Portefeuille au {comp_date.date()} : {len(comp_df)} positions")
if comp_df.empty or 'Weight' not in comp_df.columns:
    print("  Composition indisponible pour cette date (probablement hors rebalancement).")
else:
    print(f"  Long  : {(comp_df['Weight'] > 0).sum()} positions")
    print(f"  Short : {(comp_df['Weight'] < 0).sum()} positions")
    print()
    display(comp_df.head())


Portefeuille au 2026-01-30 : 470 positions
  Long  : 224 positions
  Short : 235 positions



,Ticker,Name,Country,Currency,Sector,Industry,Price,Weight
0,ABVX FP Equity,ABIVAX SA,FRANCE,EUR,Soins de santé,Biotechnologie,94.80,0.033118
1,FRES LN Equity,FRESNILLO PLC,MEXICO,GBp,Matériaux,Métaux et mines,3702.00,0.023091
2,HOT GY Equity,HOCHTIEF AG,GERMANY,EUR,Industrie,Construction et ingénierie,354.80,0.017630
3,JDEP NA Equity,JDE PEET'S NV,NETHERLANDS,EUR,Consommation de base,Produits alimentaires,31.62,0.016565
4,ZEG LN Equity,ZEGONA COMMUNICATIONS PLC,BRITAIN,GBp,Services de communication,Services de télécommunicatio,1575.00,0.016432


In [ ]:
comp_charts = reporter.plot_composition_barcharts(comp_date)

comp_charts["Sector"].show()

In [ ]:
comp_charts["Country"].show()

In [ ]:
comp_charts["Industry"].show()

In [ ]:
comp_charts["Currency"].show()

## 6. Analyse du portefeuille

### 6a. Top 10 positions 

In [ ]:
top10_weights = reporter.get_top_10_weights(comp_date)

print(f"Top 10 positions au {comp_date.date()} :")
display(top10_weights)

Top 10 positions au 2026-01-30 :


,Ticker,Name,Sector,Country,Weight
0,ABVX FP Equity,ABIVAX SA,Soins de santé,FRANCE,0.033118
1,FRES LN Equity,FRESNILLO PLC,Matériaux,MEXICO,0.023091
2,HOT GY Equity,HOCHTIEF AG,Industrie,GERMANY,0.017630
3,JDEP NA Equity,JDE PEET'S NV,Consommation de base,NETHERLANDS,0.016565
4,ZEG LN Equity,ZEGONA COMMUNICATIONS PLC,Services de communication,BRITAIN,0.016432
5,IDR SQ Equity,INDRA SISTEMAS SA,Technologies de l'information,SPAIN,0.015664
6,NDX1 GY Equity,NORDEX SE,Industrie,GERMANY,0.015500
7,AAF LN Equity,AIRTEL AFRICA PLC,Services de communication,BRITAIN,0.012904
8,FTK GY Equity,FLATEXDEGIRO SE,Finance,GERMANY,0.012862
9,UTG LN Equity,UNITE GROUP PLC/THE,Immobilier,BRITAIN,-0.012789


### 6b. Top 10 contributions à la performance

In [ ]:
top5_pos, top5_neg = reporter.get_top_10_return_contribution(comp_date)

print(f"Top 5 contributeurs POSITIFS (période {comp_date.date()}) :")
display(top5_pos)

print(f"Top 5 contributeurs NÉGATIFS (période {comp_date.date()}) :")
display(top5_neg)

Top 5 contributeurs POSITIFS (période 2026-01-30) :


,Ticker,Contribution,Name,Sector
0,FRES LN Equity,0.003408,FRESNILLO PLC,Matériaux
1,BAB LN Equity,0.003367,BABCOCK INTL GROUP PLC,Industrie
2,ZEG LN Equity,0.003317,ZEGONA COMMUNICATIONS PLC,Services de communication
3,IDR SQ Equity,0.003049,INDRA SISTEMAS SA,Technologies de l'information
4,SBMO NA Equity,0.002618,SBM OFFSHORE NV,Energie


Top 5 contributeurs NÉGATIFS (période 2026-01-30) :


,Ticker,Contribution,Name,Sector
0,ABVX FP Equity,-0.008034,ABIVAX SA,Soins de santé
1,INPST NA Equity,-0.004799,INPOST SA,Industrie
2,BEZ LN Equity,-0.004020,BEAZLEY PLC,Finance
3,LOTB BB Equity,-0.003253,LOTUS BAKERIES,Consommation de base
4,KER FP Equity,-0.002435,KERING,Consommation discrétionnaire


### 6c. Top 10 contributions au risque du portefeuille (MCTR)

In [ ]:
top5_risk_pos, top5_risk_neg = reporter.get_top_10_risk_contribution(comp_date)

print(f"Top 5 contributeurs POSITIFS au risque au {comp_date.date()} :")
display(top5_risk_pos)

print(f"Top 5 contributeurs NÉGATIFS au risque au {comp_date.date()} :")
display(top5_risk_neg)

Top 5 contributeurs POSITIFS au risque au 2026-01-30 :


,Ticker,Risk Contribution,Weight,Name,Sector
0,ABVX FP Equity,0.146703,0.033118,ABIVAX SA,Soins de santé
1,FRES LN Equity,0.002071,0.023091,FRESNILLO PLC,Matériaux
2,HOT GY Equity,0.001745,0.017630,HOCHTIEF AG,Industrie
3,ENR GY Equity,0.001606,0.009622,SIEMENS ENERGY AG,Industrie
4,NDX1 GY Equity,0.001440,0.015500,NORDEX SE,Industrie


Top 5 contributeurs NÉGATIFS au risque au 2026-01-30 :


,Ticker,Risk Contribution,Weight,Name,Sector
0,HIAB FH Equity,-0.000387,-0.003772,HIAB OYJ,Industrie
1,VALMT FH Equity,-0.000296,-0.002741,VALMET OYJ,Industrie
2,DTG GY Equity,-0.000263,-0.004944,DAIMLER TRUCK HOLDING AG,Industrie
3,ELI BB Equity,-0.000246,0.008342,ELIA GROUP SA/NV,Services aux collectivités
4,SPL PW Equity,-0.000236,-0.003739,SANTANDER BANK POLSKA SA,Finance


### 6d. Rendements cumulés calendaires du portefeuille

In [ ]:
fig_calendar_returns = reporter.plot_calendar_returns_heatmap()

fig_calendar_returns.show()


## 7. Portfolio PnL

PnL cumulé du portefeuille

In [ ]:
fig_pnl = reporter.plot_pnl()
fig_pnl.show()

## Exportation des Résultats

In [ ]:
out_path = reporter.run_full_report(output_dir=None)
print("Reporting folder:", out_path)

out_dir = Path(out_path)

nav_files = sorted(out_dir.glob("nav_MomentumDual_*.parquet"))

metrics_files = sorted(out_dir.glob("metriques_MomentumDual_*.parquet"))

comp_files = sorted(out_dir.glob("composition_detaillee_MomentumDual_*.parquet"))

bbu_files = sorted(out_dir.glob("bbu_MomentumDual_*.csv"))


[ReportingEngine] Génération du reporting dans : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_SignalWeight_20161230_20260220
  → Calcul des métriques...
[Report] Métriques exportées : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_SignalWeight_20161230_20260220/metriques_MomentumDual_SignalWeight.parquet
[Report] BBU CSV exporté : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_SignalWeight_20161230_20260220/bbu_MomentumDual_SignalWeight.csv
[Report] Composition détaillée exportée : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_SignalWeight_20161230_20260220/composition_detaillee_MomentumDual_SignalWeight.parquet
[Report] NAV exportée : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_SignalWeight_20161230_20260220/nav_MomentumDual_SignalWeight.parquet
[ReportingEngine] Reporting complet généré dans : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/Mo

In [ ]:

comp_files = sorted(out_dir.glob("composition_detaillee_MomentumDual_*.parquet"))
comp_df = pd.read_parquet(comp_files[0])
comp_df = pd.DataFrame(comp_df) 
comp_df["Date"] = pd.to_datetime(comp_df["Date"]) 
comp_files_df = comp_df[comp_df["Date"] == pd.Timestamp(AS_OF_DATE)] 
display(comp_files_df)

,Date,Ticker,Name,Country,Currency,Sector,Industry,Price,Weight
47280,2026-01-30,ABVX FP Equity,ABIVAX SA,FRANCE,EUR,Soins de santé,Biotechnologie,94.80,0.033118
47281,2026-01-30,FRES LN Equity,FRESNILLO PLC,MEXICO,GBp,Matériaux,Métaux et mines,3702.00,0.023091
47282,2026-01-30,HOT GY Equity,HOCHTIEF AG,GERMANY,EUR,Industrie,Construction et ingénierie,354.80,0.017630
47283,2026-01-30,JDEP NA Equity,JDE PEET'S NV,NETHERLANDS,EUR,Consommation de base,Produits alimentaires,31.62,0.016565
47284,2026-01-30,ZEG LN Equity,ZEGONA COMMUNICATIONS PLC,BRITAIN,GBp,Services de communication,Services de télécommunicatio,1575.00,0.016432
...,...,...,...,...,...,...,...,...,...
47745,2026-01-30,BME LN Equity,B&M EUROPEAN VALUE RETAIL SA,BRITAIN,GBp,Consommation discrétionnaire,Distributeurs - Diversifiés,176.35,-0.010796
47746,2026-01-30,EDEN FP Equity,EDENRED,FRANCE,EUR,Finance,Services financiers,17.67,-0.012208
47747,2026-01-30,PNDORA DC Equity,PANDORA A/S,DENMARK,DKK,Consommation discrétionnaire,"Textiles, habillement et produ",509.00,-0.012442
47748,2026-01-30,ORSTED DC Equity,ORSTED A/S,DENMARK,DKK,Services aux collectivités,Producteurs indépendants d'é,141.60,-0.012629


In [ ]:
bbu_df = pd.read_csv(bbu_files[0])
display(bbu_df.head())

,Date,Ticker,Weights
0,2016-12-30,KESKOB FH Equity,0.013027
1,2016-12-30,MRW LN Equity,0.012053
2,2016-12-30,TSCO LN Equity,0.011633
3,2016-12-30,MOWI NO Equity,0.007857
4,2016-12-30,JMT PL Equity,0.006148
